# 📘 Notebook: Chapter 4 - Design Principles & Best Practices

This notebook accompanies Chapter 4, “Design Principles & Best Practices,” and provides hands-on demonstrations of the core mechanisms that support reliable, controllable, and reproducible prompt engineering. It complements the chapter by showing how experimental workflows in Jupyter Notebooks can be used to evaluate reproducibility, test structural constraints, and validate output-control techniques prior to deployment.

### What this notebook covers

**Reproducibility Experiments:**  
Compare deterministic vs. stochastic sampling (temperature, top-p), run multiple iterations of the same prompt, and analyze which parts of the output remain stable or variable. Using a free Hugging Face model (GPT-2), the notebook explores how fixed templates differ from loosely structured prompts in formatting consistency.

**Template & Formatting Behavior:**  
Examine how models respond to structured templates versus flexible instructions, enabling detection of formatting drift, stability issues, and model sensitivity to prompt structure.

**Output Control & Structural Enforcement:**  
Demonstrate strict-format prompting, JSON-style schema constraints, and Python-based validation to detect malformed outputs. Includes examples of prompting for corrected responses and separating reasoning from final output formatting.





# Part I – Reproducibility Experiments with Hugging Face Models

We will:

- Compare **deterministic** vs **stochastic** sampling (temperature, top_p)
- Log and inspect multiple runs of the **same prompt**
- Observe which parts of the output are stable vs variable
- Compare **fixed templates** vs **loosely structured prompts** in terms of formatting stability

We use a free Hugging Face model (`gpt2`) so that anyone can run this notebook without an API key.


## 1. Install Required Libraries

These are all free and work on CPU or GPU.

In [1]:
!pip install --quiet transformers accelerate pandas torch

## 2. Load the Hugging Face Model and Tokenizer

We load the GPT2 model and tokenizer from Hugging Face.  
GPT2 is small enough to run for free and demonstrates reproducibility concepts clearly.

It also selects the correct device (CPU or GPU) and ensures that a padding token exists.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import time

# Choose a small, free model from Hugging Face
MODEL_NAME = "gpt2"  # You can later try "gpt2-medium" or an instruct model if you have more resources

# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

# Some models have no pad token; fall back to eos token if so
if tokenizer.pad_token_id is None:
  tokenizer.pad_token = tokenizer.eos_token


Using device: cpu


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## 3. Define the Base Prompt and the Text Generation Function

Here we set up:

- A **base prompt** that will be reused in experiments  
- A **helper function** that generates text from the model under both deterministic and stochastic conditions  

The function supports:

- Temperature  
- Top p sampling  
- Deterministic decoding (do_sample=False)  
- Removing the original prompt text for easier comparison  

This setup enables controlled reproducibility testing.


In [3]:
BASE_PROMPT = (
    "Explain the concept of opportunity cost to a high school student in about 150 words. "
    "Use clear language and one concrete example."
)

def generate_text(
    prompt: str,
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_new_tokens: int = 150,
    do_sample: bool = False,
) -> str:
    """
    Generate text from the model with specified sampling parameters.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Remove the prompt prefix so we see only the model's continuation
    if full_text.startswith(prompt):
        return full_text[len(prompt):].strip()
    return full_text.strip()


## 4. Run Deterministic and Stochastic Sampling Experiments

Here we run the same prompt multiple times under two decoding schemes:

### Deterministic:
- Temperature = 0  
- do_sample = False  
- Produces stable or identical results

### Stochastic:
- Temperature = 1.0  
- top p = 0.9  
- do_sample = True  
- Produces varied results

The outputs are collected in a DataFrame for comparison.


In [16]:
def run_experiment(
    n_runs: int = 5,
    det_temperature: float = 0.0,
    stoch_temperature: float = 1.0,
    stoch_top_p: float = 0.9,
):
    """
    Runs the same prompt multiple times under deterministic and stochastic settings.
    Returns a DataFrame with all outputs.
    """

    rows = []

    def safe_generate(prompt, **gen_kwargs):
        """
        Wrapper that only uses parameters supported by the model.
        """
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # Filter out unsupported arguments
        supported = {
            k: v for k, v in gen_kwargs.items()
            if k in model.generation_config.to_dict()
        }

        output_ids = model.generate(**inputs, **supported)
        return tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # -------------------------
    # Deterministic runs
    # -------------------------
    for i in range(n_runs):
        text = safe_generate(
            BASE_PROMPT,
            temperature=det_temperature,
            top_p=1.0,
            do_sample=False,
        )
        rows.append({
            "run_id": i + 1,
            "mode": "deterministic",
            "temperature": det_temperature,
            "top_p": 1.0,
            "output": text,
        })
        time.sleep(0.2)

    # -------------------------
    # Stochastic runs
    # -------------------------
    for i in range(n_runs):
        text = safe_generate(
            BASE_PROMPT,
            temperature=stoch_temperature,
            top_p=stoch_top_p,
            do_sample=True,
        )
        rows.append({
            "run_id": i + 1,
            "mode": "stochastic",
            "temperature": stoch_temperature,
            "top_p": stoch_top_p,
            "output": text,
        })
        time.sleep(0.2)

    return pd.DataFrame(rows)


df_outputs = run_experiment(n_runs=5)
df_outputs.head()


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,run_id,mode,temperature,top_p,output
0,1,deterministic,0.0,1.0,\nYou are a risk analysis assistant.\n\nAnalyz...
1,2,deterministic,0.0,1.0,\nYou are a risk analysis assistant.\n\nAnalyz...
2,3,deterministic,0.0,1.0,\nYou are a risk analysis assistant.\n\nAnalyz...
3,4,deterministic,0.0,1.0,\nYou are a risk analysis assistant.\n\nAnalyz...
4,5,deterministic,0.0,1.0,\nYou are a risk analysis assistant.\n\nAnalyz...


## 5. Compare Output Stability and Variation

We then compare the outputs of deterministic and stochastic runs.

It prints:

- The number of unique outputs for each mode  
- Each output in full for qualitative analysis  

This demonstrates:

- Deterministic outputs stay stable  
- Stochastic outputs vary in wording, order, and phrasing  


In [5]:
# How many unique outputs per mode?
unique_counts = df_outputs.groupby("mode")["output"].nunique()
print("Unique outputs per mode:\n", unique_counts, "\n")

print("=== Deterministic outputs ===\n")
for _, row in df_outputs[df_outputs["mode"] == "deterministic"].iterrows():
    print(f"Run {row['run_id']}:\n{row['output']}\n{'-'*80}\n")

print("=== Stochastic outputs ===\n")
for _, row in df_outputs[df_outputs["mode"] == "stochastic"].iterrows():
    print(f"Run {row['run_id']}:\n{row['output']}\n{'-'*80}\n")


Unique outputs per mode:
 mode
deterministic    1
stochastic       5
Name: output, dtype: int64 

=== Deterministic outputs ===

Run 1:
The idea of a high school student being able to afford to attend a college is a very good one. It's a good idea to have a high school student who is able to afford to attend college.

The idea of a high school student being able to afford to attend college is a very good one. It's a good idea to have a high school student who is able to afford to attend college.

The idea of a high school student being able to afford to attend college is a very good one. It's a good idea to have a high school student who is able to afford to attend college.

The idea of a high school student being able to afford to attend college is a very good one. It
--------------------------------------------------------------------------------

Run 2:
The idea of a high school student being able to afford to attend a college is a very good one. It's a good idea to have a high scho

# 6. Comparing Fixed Templates vs Loosely Structured Prompts

This section examines how different prompt designs influence reproducibility.

### Template Prompt:
A structured format with explicit constraints and required sections.

### Loose Prompt:
An open ended natural language instruction with minimal structure.

We will compare:

- Formatting consistency  
- Heading stability  
- Variability in text structure  


## 7. Define Template Style and Loosely Structured Prompts

We then define two prompt styles:

- **Template Prompt**: Includes headings, constraints, and explicit formatting expectations.  
- **Loose Prompt**: Only gives a general instruction.

A helper function generates model outputs under stochastic sampling so differences become visible.


In [6]:
TEMPLATE_PROMPT = """
Task:
Explain the concept of opportunity cost to a high school student.

Constraints:
- Length between 120 and 160 words.
- Use exactly one concrete real world example.
- Use a short section structure with two headings: "Definition" and "Example".

Output format:
Definition:
[one short paragraph]

Example:
[one short paragraph]
""".strip()

LOOSE_PROMPT = (
    "Explain opportunity cost to a high school student and give an example. "
    "Keep it fairly short, but detailed enough to be useful."
)

def generate_from_prompt(
    prompt: str,
    temperature: float = 1.0,
    top_p: float = 0.9,
    max_new_tokens: int = 180,
) -> str:
    text = generate_text(
        prompt,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens,
        do_sample=True,
    )
    return text


## 8. Run Template Based and Loose Prompt Experiments

The following code snippet runs several stochastic generations for each prompt type and stores the outputs.

By comparing multiple runs, we can observe:

- How stable the template based formatting remains  
- How loosely structured prompts drift in structure and phrasing  


In [7]:
def compare_template_vs_loose(n_runs: int = 5):
    rows = []

    # Template based runs
    for i in range(n_runs):
        out_template = generate_from_prompt(TEMPLATE_PROMPT)
        rows.append(
            {
                "run_id": i + 1,
                "prompt_type": "template",
                "output": out_template,
            }
        )
        time.sleep(0.2)

    # Loose prompt runs
    for i in range(n_runs):
        out_loose = generate_from_prompt(LOOSE_PROMPT)
        rows.append(
            {
                "run_id": i + 1,
                "prompt_type": "loose",
                "output": out_loose,
            }
        )
        time.sleep(0.2)

    return pd.DataFrame(rows)


df_template_loose = compare_template_vs_loose(n_runs=5)
df_template_loose.head()


,run_id,prompt_type,output
0,1,template,Example:\n\n[one long paragraph]\n\nExample:\n...
1,2,template,The idea of opportunity cost was invented by W...
2,3,template,The second headings and the first headings can...
3,4,template,[one short paragraph]\n\n[one short paragraph]...
4,5,template,"(If you're not familiar with ""Example"", use th..."


## 9. Inspect Formatting Drift and Structural Stability

We'll then print:

- The number of unique outputs for each prompt style  
- All generated outputs for side by side analysis  

Template prompts generally maintain:

- Consistent headings  
- Robust structure  
- Predictable formatting  

Loose prompts often exhibit:

- Inconsistent paragraphing  
- Missing or extra details  
- Shifts in tone or length  

This demonstrates how structure improves reproducibility.


In [8]:
print("Unique outputs per prompt type:\n")
print(df_template_loose.groupby("prompt_type")["output"].nunique(), "\n")

print("=== Template based outputs ===\n")
for _, row in df_template_loose[df_template_loose["prompt_type"] == "template"].iterrows():
    print(f"Run {row['run_id']}:\n{row['output']}\n{'-'*80}\n")

print("=== Loose prompt outputs ===\n")
for _, row in df_template_loose[df_template_loose["prompt_type"] == "loose"].iterrows():
    print(f"Run {row['run_id']}:\n{row['output']}\n{'-'*80}\n")


Unique outputs per prompt type:

prompt_type
loose       5
template    5
Name: output, dtype: int64 

=== Template based outputs ===

Run 1:
Example:

[one long paragraph]

Example:

[one short paragraph]

Output format:

Example:

[one short paragraph]

Example:

[two short paragraphs]

Example:

[one long paragraph]

Example:

[two long paragraphs]

Example:

[two long paragraphs]

Example:

[one short paragraph]

Example:

[one long paragraph]

Example:

[two short paragraphs]

Example:

[one long paragraph]

Example:

[one long paragraph]

Example:

[one short paragraph]

Example:

[one long paragraph]

Example:

[one long paragraph]

Example:

[one short paragraph
--------------------------------------------------------------------------------

Run 2:
The idea of opportunity cost was invented by William Clark. In a number of his papers he has explored the concept of the cost of something (for example, that in the case of a student who has an education plan that includes a teacher,

# Part II –  Output Control and Structural Enforcement – JSON Validation Demo

We will:

- Ask a language model for **strict JSON output**
- **Validate** the JSON structure in Python
- Detect format drift and **request a corrected output**
- Self-Correct output via a second prompt
- Decoupling reasoning from output formatting
- Schema-style validation in Python

We use a small Hugging Face model for illustration, but in a production setting you would plug in GPT-4.x, Claude, or any frontier model that supports structured outputs.


## 1. Define and Initialize Language Model

In [9]:
MODEL_NAME = "gpt2"  # swap with a better instruct model if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


Using device: cpu


## 2. Requesting Strict JSON Output

Here we ask the model to output **only JSON**, with a simple, fixed structure:

- `risk_level` (string)
- `summary` (string)
- `actions` (list of strings)

We will then parse the output with `json.loads` and see whether it is valid.


In [10]:
import json

BASE_PROMPT = """
You are a risk analysis assistant.

Analyze the following situation and output ONLY a JSON object, with no extra text.

Situation:
A medical clinic is rolling out a new electronic health record system that integrates third-party AI tools for automated documentation.

JSON format:
{
  "risk_level": "<low|medium|high>",
  "summary": "<one-sentence summary>",
  "actions": ["<recommended action 1>", "<recommended action 2>"]
}

Now produce ONLY the JSON object.
"""

def generate_raw_output(prompt: str, max_new_tokens: int = 200) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.9,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Try to strip the prompt, if the model echoes it
    if text.startswith(prompt):
        text = text[len(prompt):]
    return text.strip()

raw_output = generate_raw_output(BASE_PROMPT)
print("Raw model output:\n", raw_output)


Raw model output:
 Situation:

A new patient with cancer receives a CT scan of the tumor.

JSON format:

{

"risk_level": "cancer",

"summary": "A cancer patient with cancer receives an MRI scan of his tumor".

}

Now produce only the JSON object.

Situation:

A nurse uses an online tool to provide a free, online, diagnostic evaluation of an unknown patient.

JSON format:

{

"risk_level": "non-cancer",

"summary": "A nurse uses an online tool to provide a free, online, diagnostic evaluation of an unknown patient".

}

Now produce only the JSON object.

Situation:

A nurse uses an online tool to provide a free, online, diagnostic evaluation of an unknown patient.

JSON format:

{

"risk_level": "non-cancer


## 3. Validating JSON and Handling Format Drift

We now:

1. Try to parse the model output as JSON.
2. If parsing fails, we detect this as **format drift**.
3. Later, we can ask the model to self-correct based on the error.


In [11]:
def parse_and_validate_json(text: str):
    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        print("❌ JSON parsing failed:", e)
        return None

    # Minimal structural checks
    required_keys = {"risk_level", "summary", "actions"}
    missing = required_keys - set(data.keys())
    if missing:
        print("❌ JSON missing required keys:", missing)
        return None

    if not isinstance(data["actions"], list):
        print("❌ 'actions' must be a list.")
        return None

    print("✅ JSON is syntactically valid and structurally consistent.")
    return data

parsed = parse_and_validate_json(raw_output)
parsed


❌ JSON parsing failed: Expecting value: line 1 column 1 (char 0)


## 4. Self-Correcting Output via a Second Prompt

If the first output is invalid, we can:

1. Send the **original prompt** and the **invalid output** back to the model.
2. Ask it to return **corrected JSON only**.
3. Re-validate and, if needed, iterate.

This pattern combines model-based self-checking with deterministic validation.


In [12]:
def correct_json_with_model(original_prompt: str, bad_output: str) -> str:
    correction_prompt = f"""
You attempted to answer the following instruction:

{original_prompt}

However, your output was NOT valid JSON:

{bad_output}

Please fix the output so that it is VALID JSON and follows the requested format exactly.
Return ONLY the corrected JSON object, nothing else.
"""

    corrected = generate_raw_output(correction_prompt, max_new_tokens=200)
    return corrected.strip()

if parsed is None:
    print("Attempting self-correction via the model...\n")
    corrected_output = correct_json_with_model(BASE_PROMPT, raw_output)
    print("Corrected output:\n", corrected_output)
    parsed_corrected = parse_and_validate_json(corrected_output)
    parsed_corrected


Attempting self-correction via the model...

Corrected output:
 Situation:

You are running out of space.

JSON format:

{

"risk_level": "no-cancer",

"summary": "You are running out of space".

}

Now generate ONLY the JSON object.

Situation:

A hospital wants a patient to take a shot at a hospital.

JSON format:

{

"risk_level": "no-cancer",

"summary": "A hospital wants a patient to take a shot at a hospital.

Return ONLY the correct JSON object, nothing else.

Situation:

You are not using a scanner.

JSON format:

{

"risk_level": "no-cancer",

"summary": "You are not using a scanner.

return ONLY the correct JSON object, nothing else.

Situation:

You have a scan of
❌ JSON parsing failed: Expecting value: line 1 column 1 (char 0)


## 5. Decoupling Reasoning from Output Formatting

To mirror the “Thinking-First” architecture:

1. We let the model **think freely** inside `<thinking>` tags.
2. We then request a **separate, clean JSON block** as the final output.
3. Only the JSON block is parsed and sent downstream.

Here we simulate this pattern (in practice, frontier models would hide the thinking from the user).


In [13]:
THINKING_PROMPT = """
You are a legal risk assessment assistant.

First, think step-by-step about the risk in <thinking> tags.
Then, output ONLY a JSON object with the following fields:
- risk_level: "low", "medium", or "high"
- summary: one-sentence summary
- actions: list of concrete recommended actions

Document:
The contract delegates data processing to a third-party vendor without a clear DPA, and cross-border transfers are implied but not explicitly regulated.

Format example:
<thinking>
[...reasoning here...]
</thinking>
{
  "risk_level": "...",
  "summary": "...",
  "actions": ["...", "..."]
}

Now produce your answer.
"""

raw = generate_raw_output(THINKING_PROMPT, max_new_tokens=300)
print(raw)


[...]

<thinking>

[...reasoning here...]

</thinking>

The contract delegates data processing to a third-party vendor without a clear DPA, and cross-border transfers are implied but not explicitly regulated.

Format example:

<thinking>

[...reasoning here...]

</thinking>

{

- risk_level: "low",

- summary: "...",

- actions: "list of concrete recommended actions",

- actions: "list of concrete recommended actions",

-

-

-

-

-

[...]

</thinking>

[...reasoning here...]

</thinking>

{

- risk_level: "medium",

- summary: "...",

- actions: {"sitemap"}

}

Output your answer.

[...]

<thinking>

[...reasoning here...]

</thinking>

</thinking>

<thinking>

[...reasoning here...]

</thinking>

{

- risk_level: "high",

- summary: "...",

- actions: {"sitemap"}

}

Output your answer.

[...]

<thinking>

[...reasoning here...]


## 6. Extracting and Validating the Final Structured Block

We now:

- Ignore the `<thinking>` section.
- Extract the JSON object from the rest.
- Run our existing validator on the extracted JSON only.


In [14]:
import re

def extract_json_block(text: str) -> str | None:
    # Very simple heuristic: find the first { ... } block
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        print("❌ No JSON block found.")
        return None
    return match.group(0).strip()

json_block = extract_json_block(raw)
if json_block:
    print("Extracted JSON block:\n", json_block, "\n")
    parsed = parse_and_validate_json(json_block)
    parsed


Extracted JSON block:
 {

- risk_level: "low",

- summary: "...",

- actions: "list of concrete recommended actions",

- actions: "list of concrete recommended actions",

-

-

-

-

-

[...]

</thinking>

[...reasoning here...]

</thinking>

{

- risk_level: "medium",

- summary: "...",

- actions: {"sitemap"}

}

Output your answer.

[...]

<thinking>

[...reasoning here...]

</thinking>

</thinking>

<thinking>

[...reasoning here...]

</thinking>

{

- risk_level: "high",

- summary: "...",

- actions: {"sitemap"}

} 

❌ JSON parsing failed: Expecting property name enclosed in double quotes: line 3 column 1 (char 3)


## 7. Schema-Style Validation in Python

This is not full Grammar-Constrained Decoding,
but it shows how to:

- Define a simple "schema" in Python
- Check the model output against it
- Mimic the idea of **native structured outputs**

We assume the JSON has already been parsed.


In [15]:
EXPECTED_SCHEMA = {
    "risk_level": {"type": "string", "allowed": ["low", "medium", "high"]},
    "summary": {"type": "string"},
    "actions": {"type": "list", "item_type": "string"},
}

def validate_schema(data: dict, schema: dict) -> bool:
    # Check required keys
    for key, spec in schema.items():
        if key not in data:
            print(f"❌ Missing key: {key}")
            return False

        value = data[key]
        if spec["type"] == "string":
            if not isinstance(value, str):
                print(f"❌ Key '{key}' must be a string.")
                return False
            if "allowed" in spec and value not in spec["allowed"]:
                print(f"❌ Key '{key}' has invalid value: {value}")
                return False

        elif spec["type"] == "list":
            if not isinstance(value, list):
                print(f"❌ Key '{key}' must be a list.")
                return False
            if spec.get("item_type") == "string":
                if not all(isinstance(x, str) for x in value):
                    print(f"❌ All elements in '{key}' must be strings.")
                    return False

    print("✅ Output satisfies the expected schema.")
    return True

if parsed:
    validate_schema(parsed, EXPECTED_SCHEMA)


## 8. Schema Validation Tests

The following tests demonstrate how to verify that a model’s structured output matches the expected schema.  
We run several examples, including valid responses and intentionally flawed ones, to illustrate how the validator detects:

- Missing fields  
- Incorrect types  
- Invalid enum values  
- Malformed list items  

These tests help ensure that downstream systems receive consistently structured, machine-readable outputs, even when models vary in how they generate content.


In [18]:
parsed = {
    "risk_level": "medium",
    "summary": "The system is stable but requires monitoring.",
    "actions": ["review logs", "schedule follow-up"],
}

validate_schema(parsed, EXPECTED_SCHEMA)


✅ Output satisfies the expected schema.


True

In [19]:
parsed = {
    "risk_level": "low",
    "summary": "Everything is normal.",
    # "actions" is missing
}

validate_schema(parsed, EXPECTED_SCHEMA)


❌ Missing key: actions


False

In [20]:
parsed = {
    "risk_level": "critical",    # not allowed
    "summary": "Unexpected value detected.",
    "actions": ["restart service"],
}

validate_schema(parsed, EXPECTED_SCHEMA)


❌ Key 'risk_level' has invalid value: critical


False

In [21]:
parsed = {
    "risk_level": "high",
    "summary": "Failure detected in subsystem.",
    "actions": ["notify team", 123],   # 123 is not a string
}

validate_schema(parsed, EXPECTED_SCHEMA)


❌ All elements in 'actions' must be strings.


False